# PSM Memory LOCOMO ingestion on Colab

This notebook is for resumable LOCOMO ingestion only. It installs the published PSM packages, prepares the model with `psm-memory setup`, restores any checkpointed database from Google Drive, and ingests turns while copying the SQLite DB and progress file back to Drive after each batch.

If Colab crashes, rerun this notebook. The next run restores the last checkpoint from Drive and continues from the saved progress index.

Recommended Colab runtime: GPU. Set `BATCH_SIZE = 1` if you want a checkpoint after every turn.

In [ ]:
# Install Node.js 22. Colab's default Node can be older than the package engine range.
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Mount Google Drive for checkpoints and keep PSM caches under /content.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import shutil

GDRIVE_DIR = "/content/drive/MyDrive/psm-memory-locomo-ingest"
WORK_DIR = "/content/psm-memory-work"
RESULT_DIR = "/content/locomo/results"
DATA = "/content/PSM/benchmark/locomo/data/locomo10.json"
MODEL = "/content/psm-memory-cache/psm-memory-qwen-1.5b-q4_k_m.gguf"
DB = "/content/locomo/results/locomo-psm-memory-ingest.db"
CHECKPOINT_DIR = f"{GDRIVE_DIR}/checkpoints"
PROGRESS = f"{GDRIVE_DIR}/locomo-ingest-progress.json"
LIMIT = 0
BATCH_SIZE = 1
WINDOW_SIZE = 2

%env PSM_MEMORY_SKIP_SETUP=1
%env PSM_MEMORY_HOME=/content/psm-memory-cache
!mkdir -p "$WORK_DIR" "$RESULT_DIR" "$GDRIVE_DIR" "$CHECKPOINT_DIR"
%cd /content/psm-memory-work
!npm init -y >/dev/null 2>&1 || true
!npm install @psm-memory/cli@latest @psm-memory/sdk@latest
!rm -rf /content/PSM
!git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
!cp /content/PSM/benchmark/locomo/colab/locomo-benchmark.mjs /content/psm-memory-work/locomo-benchmark.mjs
!ls -lh /content/PSM/benchmark/locomo/data/locomo10.json /content/psm-memory-work/locomo-benchmark.mjs

In [ ]:
# Restore the latest checkpointed DB from Drive before resuming ingestion.
checkpoint_db = Path(CHECKPOINT_DIR) / Path(DB).name
if checkpoint_db.exists():
    Path(DB).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint_db, DB)
    for suffix in ("-wal", "-shm"):
        sidecar = Path(str(checkpoint_db) + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, str(DB) + suffix)
    print(f"Restored DB checkpoint from {checkpoint_db}")
else:
    print("No DB checkpoint found in Drive yet.")

if Path(PROGRESS).exists():
    progress = json.loads(Path(PROGRESS).read_text())
    print(f"Progress file found: next_index={progress.get('next_index', 0)}")
else:
    print("No progress file found yet.")

In [ ]:
# Prepare the default model once. Embeddings are skipped for ingestion-only runs.
!npx psm-memory setup --skip-embeddings --yes --pretty

In [ ]:
# Resumable ingest. Every batch writes the DB and progress file into Google Drive.
!node /content/psm-memory-work/locomo-benchmark.mjs ingest --data "$DATA" --db "$DB" --model "$MODEL" --limit $LIMIT --batch-size $BATCH_SIZE --window-size $WINDOW_SIZE --progress "$PROGRESS" --checkpoint-dir "$CHECKPOINT_DIR"

In [ ]:
# Inspect local and Drive artifacts.
!ls -lh /content/locomo/results
!ls -lh "$GDRIVE_DIR" "$CHECKPOINT_DIR"
summary = Path('/content/locomo/results/ingest-summary.json')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2)[:4000])

If you want one checkpoint per turn, keep `BATCH_SIZE = 1`. For faster but less granular recovery, raise it to a larger batch size. The resume point is tracked in `locomo-ingest-progress.json` and the SQLite checkpoint lives under `checkpoints/` in Google Drive.